# 02 — Point in Polygon: The Question

Before we write a single line of algorithm, let's be clear about what we're actually asking.

Given a point and a polygon:

```text
Is this point inside that polygon?
```

For a square, your brain answers instantly. For an irregular shape — one with concave notches, jagged edges, or a complex outline — your brain can't always answer confidently. Neither can a naive algorithm. This notebook builds the intuition we need before implementing anything.

## Setup

In [ ]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, Marker, basemaps

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## Polygons Are Closed Rings

A GeoJSON Polygon is stored as a list of coordinate rings. Each ring is a list of `[lon, lat]` pairs where the first and last point are **identical** — the ring is explicitly closed.

```python
# A simple rectangle (not closed — WRONG)
ring = [[42,34], [58,34], [58,42], [42,42]]

# Closed ring (CORRECT)
ring = [[42,34], [58,34], [58,42], [42,42], [42,34]]   # first == last
```

If you forget to close the ring, ray casting silently skips the last edge — the one that connects the last vertex back to the first. Points near that edge will return wrong answers and you'll have no idea why.

In [ ]:
# Define our teaching polygon — a rough quadrilateral over the Gulf region
# Coordinates are [lon, lat] — GeoJSON convention
sector_alpha = {
    "type": "Feature",
    "properties": {"name": "Sector Alpha"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [42, 34], [58, 34], [58, 42], [42, 42], [42, 34]   # closed
        ]]
    }
}

m = Map(center=(38, 50), zoom=4, basemap=basemaps.CartoDB.Positron)
m.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [sector_alpha]},
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.2, "weight": 2}
))

ring = sector_alpha["geometry"]["coordinates"][0]
print(f"Ring has {len(ring)} points")
print(f"First point: {ring[0]}")
print(f"Last point:  {ring[-1]}")
print(f"Closed: {ring[0] == ring[-1]}")
m

## Inside vs Outside: The Easy Cases

Some points are obviously inside and some are obviously outside. Let's label them before we compute anything.

In [ ]:
# Test points — [lon, lat] order (GeoJSON convention)
test_points = [
    {"lon": 51.388, "lat": 35.695, "label": "Tehran",     "expected": "inside"},
    {"lon": 46.675, "lat": 24.688, "label": "Riyadh",     "expected": "outside"},  # below sector
    {"lon": 30.000, "lat": 38.000, "label": "Black Sea",  "expected": "outside"},  # west of sector
    {"lon": 50.000, "lat": 38.000, "label": "Center",     "expected": "inside"},
    {"lon": 70.000, "lat": 38.000, "label": "East",       "expected": "outside"},  # east of sector
]

# Visualize on a map
point_features = [
    {
        "type": "Feature",
        "properties": {"name": p["label"], "expected": p["expected"]},
        "geometry": {"type": "Point", "coordinates": [p["lon"], p["lat"]]}
    }
    for p in test_points
]

m2 = Map(center=(36, 50), zoom=4, basemap=basemaps.CartoDB.Positron)
m2.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [sector_alpha]},
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.15, "weight": 2}
))
m2.add(GeoJSON(
    data={"type": "FeatureCollection", "features": point_features},
    style={"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 1.0, "weight": 2}
))

print(f"{'Point':<15}  {'Expected':>10}")
print("─" * 30)
for p in test_points:
    print(f"  {p['label']:<13}  {p['expected']:>10}")
m2

## Why Not Just Check the Bounding Box?

A bounding box check (`min_lon ≤ lon ≤ max_lon AND min_lat ≤ lat ≤ max_lat`) seems like it would work for a rectangle. And it does — for rectangles.

But most real polygons are not rectangles. Consider a concave polygon — one that has an indentation. A point can pass the bounding box test (it's within the rectangle that surrounds the polygon) but still be *outside* the actual polygon, inside the notch.

In [ ]:
# A concave (non-convex) polygon — like a C-shape or notched region
# Coordinates: [lon, lat]
concave_poly = {
    "type": "Feature",
    "properties": {"name": "Concave Zone"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [44, 34], [60, 34], [60, 44],
            [55, 44], [55, 38], [49, 38],   # notch cut into the top-right
            [49, 44], [44, 44], [44, 34]    # closed
        ]]
    }
}

# A point inside the BOUNDING BOX but inside the NOTCH (outside the polygon)
notch_point = {
    "type": "Feature",
    "properties": {"name": "Inside notch (outside polygon)"},
    "geometry": {"type": "Point", "coordinates": [52.0, 41.0]}
}

# A point clearly inside the polygon
inside_point = {
    "type": "Feature",
    "properties": {"name": "Clearly inside"},
    "geometry": {"type": "Point", "coordinates": [47.0, 37.0]}
}

m3 = Map(center=(39, 52), zoom=5, basemap=basemaps.CartoDB.Positron)
m3.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [concave_poly]},
    style={"color": "#8e44ad", "fillColor": "#8e44ad", "fillOpacity": 0.2, "weight": 2}
))
m3.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [notch_point, inside_point]},
    style={"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 1.0, "weight": 2}
))

print("Purple polygon has a notch cut out of it.")
print("The red point in the notch PASSES bounding box test but is OUTSIDE the polygon.")
m3

## The Three Cases

Every point-in-polygon query has exactly three outcomes:

```text
1. Inside   — clearly within the boundary
2. Outside  — clearly outside the boundary
3. On edge  — exactly on the boundary line
```

Cases 1 and 2 are straightforward. Case 3 is ambiguous — and different algorithms handle it differently. The ray casting algorithm (notebook 03) treats boundary points inconsistently depending on which edge the point lands on. In practice, this rarely matters for geographic applications where coordinates have floating-point noise. But it is worth knowing.

In [ ]:
# Demonstrate boundary ambiguity
# A point exactly on the left edge of Sector Alpha: lon=42, lat=38

boundary_cases = [
    {"lon": 42.0,  "lat": 38.0,  "label": "On left edge (lon=42)"},
    {"lon": 50.0,  "lat": 34.0,  "label": "On bottom edge (lat=34)"},
    {"lon": 42.0,  "lat": 34.0,  "label": "On corner vertex"},
]

boundary_features = [
    {
        "type": "Feature",
        "properties": {"name": p["label"]},
        "geometry": {"type": "Point", "coordinates": [p["lon"], p["lat"]]}
    }
    for p in boundary_cases
]

m4 = Map(center=(38, 50), zoom=5, basemap=basemaps.CartoDB.Positron)
m4.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [sector_alpha]},
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.15, "weight": 2}
))
m4.add(GeoJSON(
    data={"type": "FeatureCollection", "features": boundary_features},
    style={"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 1.0, "weight": 2}
))

print("Red points are on the boundary of the blue polygon.")
print("These are the edge cases that can return inconsistent results.")
m4

## Building the Regions Dataset

We define five named sectors over the Middle East region. These will be used across notebooks 03–06 as our polygon test dataset. Each sector is a simple quadrilateral, with non-overlapping boundaries.

In [ ]:
sectors = [
    {
        "name": "Sector Alpha",
        "description": "Northern Gulf / Iran",
        "ring": [[42,34],[58,34],[58,42],[42,42],[42,34]],
    },
    {
        "name": "Sector Bravo",
        "description": "Arabian Peninsula",
        "ring": [[42,22],[58,22],[58,34],[42,34],[42,22]],
    },
    {
        "name": "Sector Charlie",
        "description": "Levant / Eastern Mediterranean",
        "ring": [[26,33],[42,33],[42,40],[26,40],[26,33]],
    },
    {
        "name": "Sector Delta",
        "description": "Egypt / North Africa",
        "ring": [[24,22],[42,22],[42,33],[24,33],[24,22]],
    },
    {
        "name": "Sector Echo",
        "description": "Arabian Sea Coast",
        "ring": [[56,10],[75,10],[75,24],[56,24],[56,10]],
    },
]

regions_fc = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": s["name"], "description": s["description"]},
            "geometry": {"type": "Polygon", "coordinates": [s["ring"]]}
        }
        for s in sectors
    ]
}

out_path = DATA_DIR / "regions.geojson"
with open(out_path, "w") as f:
    json.dump(regions_fc, f, indent=2)

print(f"Saved {len(sectors)} sectors → {out_path}")

# Show on map
colors = ["#e74c3c", "#e67e22", "#2980b9", "#27ae60", "#8e44ad"]
m5 = Map(center=(30, 45), zoom=3, basemap=basemaps.CartoDB.Positron)
for feat, color in zip(regions_fc["features"], colors):
    m5.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        style={"color": color, "fillColor": color, "fillOpacity": 0.2, "weight": 2}
    ))

# Known cities
cities_fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "properties": {"name": "Tehran"},
         "geometry": {"type": "Point", "coordinates": [51.388, 35.695]}},
        {"type": "Feature", "properties": {"name": "Riyadh"},
         "geometry": {"type": "Point", "coordinates": [46.675, 24.688]}},
    ]
}
m5.add(GeoJSON(
    data=cities_fc,
    style={"color": "#2c3e50", "fillColor": "#2c3e50", "fillOpacity": 1.0, "weight": 1}
))
m5

## Exercise A

Load `data/regions.geojson` and create a map showing all five sectors.

Given these test points, predict by visual inspection which sector (if any) each one falls in:

| Point | lon | lat | Predicted sector |
|---|---|---|---|
| Tehran | 51.388 | 35.695 | ? |
| Riyadh | 46.675 | 24.688 | ? |
| Cairo | 31.235 | 30.044 | ? |
| Madrid | -3.703 | 40.417 | ? |
| Muscat | 58.593 | 23.614 | ? |

Write your predictions as comments in the code. We'll verify them programmatically in notebook 03.

In [ ]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "regions.geojson") as f:
    regions = json.load(f)

test_points = [
    {"name": "Tehran",  "lon": 51.388, "lat": 35.695, "predicted": "???"},  # your prediction
    {"name": "Riyadh",  "lon": 46.675, "lat": 24.688, "predicted": "???"},
    {"name": "Cairo",   "lon": 31.235, "lat": 30.044, "predicted": "???"},
    {"name": "Madrid",  "lon": -3.703, "lat": 40.417, "predicted": "???"},
    {"name": "Muscat",  "lon": 58.593, "lat": 23.614, "predicted": "???"},
]

# Display all sectors + test points on a map
# Print your predictions as comments
# Your code here

## Exercise B

Define a **non-rectangular polygon** of your own — at least 6 vertices, with at least one concave angle (a notch or indentation).

1. Make sure the ring is properly closed (first point == last point).
2. Display it on a map.
3. Place one point that is inside the bounding box but outside the polygon (in the notch).
4. Place one point that is clearly inside the polygon.
5. Print both points' coordinates and mark which is inside and which is outside by visual inspection.

In [ ]:
from ipyleaflet import Map, GeoJSON, basemaps

# Define a non-rectangular polygon (at least 6 vertices, at least one concave angle)
# Ensure ring is closed
# Place one point in the notch (inside bbox, outside polygon)
# Place one point clearly inside
# Display on map and print coordinates
# Your code here

---

## Check Your Understanding

**1.** Why is the bounding box check insufficient for non-convex polygons?

**2.** What happens if a point lies exactly on the edge of a polygon?

```python
# No code needed — answer in your own words
```

## Next

In [03 — Ray Casting Algorithm](./03_Ray_Casting_Algorithm.ipynb), we implement the algorithm that actually answers the question — shooting a ray from the point and counting how many polygon edges it crosses.